In [ ]:
import copy
import json
import warnings
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*The distribution is specified by.*")

sns.set_theme(style="whitegrid")
target_palette = {0: "#2a9d8f", 1: "#e76f51"}
heatmap_palette = "Spectral_r"

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
cache_dir = Path("data")
cache_dir.mkdir(exist_ok=True)

x_parquet_path = cache_dir / "cdc_diabetes_X.parquet"
y_parquet_path = cache_dir / "cdc_diabetes_y.parquet"
x_csv_path = cache_dir / "cdc_diabetes_X.csv"
y_csv_path = cache_dir / "cdc_diabetes_y.csv"
variables_path = cache_dir / "cdc_diabetes_variables.csv"
metadata_path = cache_dir / "cdc_diabetes_metadata.json"


def read_cached_dataframe(parquet_path, csv_path):
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return None


def save_dataframe_cache(df, parquet_path, csv_path):
    try:
        df.to_parquet(parquet_path, index=False)
        print(f"Saved parquet cache: {parquet_path}")
    except Exception as error:
        df.to_csv(csv_path, index=False)
        print(f"Parquet cache failed ({error.__class__.__name__}); saved csv cache: {csv_path}")


X = read_cached_dataframe(x_parquet_path, x_csv_path)
y = read_cached_dataframe(y_parquet_path, y_csv_path)

if X is not None and y is not None and variables_path.exists() and metadata_path.exists():
    variables = pd.read_csv(variables_path)
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata_dict = json.load(file)
    print("Loaded CDC Diabetes dataset from local cache.")
else:
    cdc_diabetes_health_indicators = fetch_ucirepo(id=891)
    X = cdc_diabetes_health_indicators.data.features.copy()
    y = cdc_diabetes_health_indicators.data.targets.copy()
    variables = cdc_diabetes_health_indicators.variables.copy()
    metadata_dict = cdc_diabetes_health_indicators.metadata
    save_dataframe_cache(X, x_parquet_path, x_csv_path)
    save_dataframe_cache(y, y_parquet_path, y_csv_path)
    variables.to_csv(variables_path, index=False)
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata_dict, file, ensure_ascii=False, indent=2, default=str)
    print("Downloaded CDC Diabetes dataset and saved it to local cache.")

cdc_diabetes_health_indicators = SimpleNamespace(metadata=metadata_dict, variables=variables)
print(cdc_diabetes_health_indicators.metadata)
display(cdc_diabetes_health_indicators.variables)

In [ ]:
selected_features = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "GenHlth", "PhysHlth", "DiffWalk",
    "Sex", "MentHlth", "Age"
]

target = "Diabetes_binary"
optional_features = ["AnyHealthcare", "NoDocbcCost", "Education", "Income"]

In [ ]:
missing_selected = [col for col in selected_features if col not in X.columns]
if missing_selected:
    raise ValueError(f"Missing selected features in X: {missing_selected}")

eda_df = X[selected_features].copy()
eda_df[target] = y[target].to_numpy() if isinstance(y, pd.DataFrame) else y.to_numpy()

binary_features = [
    col for col in selected_features
    if eda_df[col].dropna().nunique() == 2 and set(eda_df[col].dropna().unique()).issubset({0, 1})
]
ordinal_features = [col for col in selected_features if col not in binary_features]

duplicate_rows = eda_df.duplicated().sum()
summary = pd.DataFrame({
    "metric": ["rows", "columns", "duplicate_rows_kept", "duplicate_share", "target_positive_share"],
    "value": [eda_df.shape[0], eda_df.shape[1], duplicate_rows, duplicate_rows / len(eda_df), eda_df[target].mean()],
})
display(summary)
display(eda_df.head())

In [ ]:
feature_dictionary = variables.loc[
    variables["name"].isin(selected_features + [target]),
    ["name", "role", "type", "demographic", "description", "missing_values"],
].reset_index(drop=True)
display(feature_dictionary)

quality_report = pd.DataFrame({
    "dtype": eda_df.dtypes.astype(str),
    "missing_count": eda_df.isna().sum(),
    "missing_share": eda_df.isna().mean(),
    "unique_values": eda_df.nunique(dropna=False),
}).sort_values(["missing_share", "unique_values"], ascending=[False, True])
quality_report = quality_report.join(eda_df[selected_features].agg(["min", "max"]).T)
display(quality_report)
display(eda_df[selected_features].describe().T)

In [ ]:
target_counts = eda_df[target].value_counts(dropna=False).rename_axis(target).reset_index(name="count")
target_counts["share"] = target_counts["count"] / len(eda_df)
display(target_counts)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=eda_df, x=target, hue=target, palette=target_palette, legend=False, ax=ax)
ax.set_title("Target distribution")
plt.tight_layout()
plt.show()

if binary_features:
    n_cols = 4
    n_rows = int(np.ceil(len(binary_features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, binary_features):
        sns.countplot(data=eda_df, x=col, hue=target, palette=target_palette, ax=ax)
        ax.set_title(col)
    for ax in axes[len(binary_features):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
rate_tables = []
for col in selected_features:
    tmp = eda_df.groupby(col, dropna=False)[target].agg(count="size", diabetes_rate="mean").reset_index()
    tmp = tmp.rename(columns={col: "value"})
    tmp.insert(0, "feature", col)
    rate_tables.append(tmp)
feature_target_rates = pd.concat(rate_tables, ignore_index=True)
display(feature_target_rates)

corr = eda_df[selected_features + [target]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap=heatmap_palette, center=0, annot=False, square=True, linewidths=0.3, ax=ax)
ax.set_title("Correlation matrix: selected features and target")
plt.tight_layout()
plt.show()

target_corr = corr[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False).to_frame("correlation_with_target")
display(target_corr)

In [ ]:
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
X_model_full = eda_df[selected_features].copy()
y_model = eda_df[target].astype(int).copy()

model_sample_size = 60000
if len(eda_df) > model_sample_size:
    X_model_full, _, y_model, _ = train_test_split(
        X_model_full, y_model, train_size=model_sample_size, random_state=42, stratify=y_model
    )

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_model_full, y_model, test_size=0.2, random_state=42, stratify=y_model
)

split_summary = pd.DataFrame({
    "part": ["modeling_sample", "train_full", "test_full"],
    "rows": [len(X_model_full), len(X_train_full), len(X_test_full)],
    "positive_share": [y_model.mean(), y_train.mean(), y_test.mean()],
    "duplicate_rows_kept": [
        pd.concat([X_model_full, y_model.rename(target)], axis=1).duplicated().sum(),
        pd.concat([X_train_full, y_train.rename(target)], axis=1).duplicated().sum(),
        pd.concat([X_test_full, y_test.rename(target)], axis=1).duplicated().sum(),
    ],
})
display(split_summary)

In [ ]:
feature_metric_rows = []
for col in selected_features:
    raw_score = X_train_full[col].to_numpy()
    roc_auc = roc_auc_score(y_train, raw_score)
    directional_score = -raw_score if roc_auc < 0.5 else raw_score
    direction = "lower_values_increase_risk" if roc_auc < 0.5 else "higher_values_increase_risk"
    unique_values = np.sort(pd.Series(raw_score).dropna().unique())
    threshold = 0.5 if len(unique_values) == 2 and set(unique_values).issubset({0, 1}) and roc_auc >= 0.5 else np.median(directional_score)
    y_single_pred = (directional_score >= threshold).astype(int)
    feature_metric_rows.append({
        "feature": col,
        "individual_roc_auc": roc_auc,
        "individual_roc_auc_strength": max(roc_auc, 1 - roc_auc),
        "individual_pr_auc": average_precision_score(y_train, directional_score),
        "individual_f2": fbeta_score(y_train, y_single_pred, beta=2, zero_division=0),
        "individual_recall": recall_score(y_train, y_single_pred, zero_division=0),
        "direction": direction,
    })

feature_scores_df = pd.DataFrame(feature_metric_rows).sort_values(
    ["individual_roc_auc_strength", "individual_pr_auc"], ascending=False
).reset_index(drop=True)
display(feature_scores_df)

correlation_threshold = 0.65
feature_strength = feature_scores_df.set_index("feature")["individual_roc_auc_strength"].to_dict()
train_corr_abs = X_train_full[selected_features].corr().abs()
features_to_drop = set()
correlation_decisions = []

ordered_features = feature_scores_df["feature"].tolist()
for i, feature_a in enumerate(ordered_features):
    if feature_a in features_to_drop:
        continue
    for feature_b in ordered_features[i + 1:]:
        if feature_b in features_to_drop:
            continue
        corr_value = train_corr_abs.loc[feature_a, feature_b]
        if corr_value > correlation_threshold:
            keep_feature, drop_feature = (feature_a, feature_b) if feature_strength[feature_a] >= feature_strength[feature_b] else (feature_b, feature_a)
            features_to_drop.add(drop_feature)
            correlation_decisions.append({
                "feature_a": feature_a,
                "feature_b": feature_b,
                "abs_corr": corr_value,
                "kept": keep_feature,
                "dropped": drop_feature,
                "kept_auc_strength": feature_strength[keep_feature],
                "dropped_auc_strength": feature_strength[drop_feature],
            })

corr_pruned_features = [col for col in selected_features if col not in features_to_drop]
correlation_decisions_df = pd.DataFrame(correlation_decisions)
display(correlation_decisions_df)

In [ ]:
fs_metric = "pr_auc"
fs_min_improvement = 0.001
fs_min_features = min(2, len(corr_pruned_features))

X_fs_train, X_fs_val, y_fs_train, y_fs_val = train_test_split(
    X_train_full[corr_pruned_features], y_train, test_size=0.25, random_state=42, stratify=y_train
)

candidate_order = feature_scores_df[feature_scores_df["feature"].isin(corr_pruned_features)]["feature"].tolist()
selected_forward_features = []
remaining_features = candidate_order.copy()
forward_rows = []
best_metric_value = -np.inf

while remaining_features:
    trial_rows = []
    for candidate in remaining_features:
        trial_features = selected_forward_features + [candidate]
        selector_model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ])
        selector_model.fit(X_fs_train[trial_features], y_fs_train)
        val_score = selector_model.predict_proba(X_fs_val[trial_features])[:, 1]
        val_pred = (val_score >= 0.5).astype(int)
        trial_rows.append({
            "candidate": candidate,
            "features": trial_features,
            "pr_auc": average_precision_score(y_fs_val, val_score),
            "roc_auc": roc_auc_score(y_fs_val, val_score),
            "f2": fbeta_score(y_fs_val, val_pred, beta=2, zero_division=0),
            "recall": recall_score(y_fs_val, val_pred, zero_division=0),
        })
    trial_df = pd.DataFrame(trial_rows).sort_values(fs_metric, ascending=False)
    best_trial = trial_df.iloc[0]
    improvement = best_trial[fs_metric] - best_metric_value
    if improvement < fs_min_improvement and len(selected_forward_features) >= fs_min_features:
        break
    selected_forward_features = best_trial["features"]
    remaining_features.remove(best_trial["candidate"])
    best_metric_value = best_trial[fs_metric]
    forward_rows.append({
        "step": len(selected_forward_features),
        "added_feature": best_trial["candidate"],
        "n_features": len(selected_forward_features),
        "validation_pr_auc": best_trial["pr_auc"],
        "validation_roc_auc": best_trial["roc_auc"],
        "validation_f2": best_trial["f2"],
        "validation_recall": best_trial["recall"],
    })

forward_selection_df = pd.DataFrame(forward_rows)
modeling_features = selected_forward_features if selected_forward_features else corr_pruned_features
X_model = X_model_full[modeling_features].copy()
X_train = X_train_full[modeling_features].copy()
X_test = X_test_full[modeling_features].copy()
print(f"Final modeling features: {len(modeling_features)}")
print(modeling_features)
display(forward_selection_df)

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, class_weight="balanced", random_state=42),
    "Naive Bayes": GaussianNB(),
    "k-NN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1))]),
    "Linear SVM": Pipeline([("scaler", StandardScaler()), ("model", LinearSVC(class_weight="balanced", dual="auto", max_iter=10000, tol=1e-3, random_state=42))]),
}

model_results = []
fitted_models = {}
predictions = {}
for name, estimator in models.items():
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    if hasattr(fitted, "predict_proba"):
        y_score = fitted.predict_proba(X_test)[:, 1]
    elif hasattr(fitted, "decision_function"):
        y_score = fitted.decision_function(X_test)
    else:
        y_score = y_pred
    fitted_models[name] = fitted
    predictions[name] = {"pred": y_pred, "score": y_score}
    model_results.append({
        "model": name,
        "train_rows": len(X_train),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
    })

results_df = pd.DataFrame(model_results).sort_values("f1", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
metric_cols = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
results_long = results_df.melt(id_vars="model", value_vars=metric_cols, var_name="metric", value_name="score")

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=results_long, x="metric", y="score", hue="model", palette="tab10", ax=ax)
ax.set_title("Model comparison on test set")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=30)
ax.legend(title="model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["model"]
print(f"Best model by F1: {best_model_name}")
print(classification_report(y_test, predictions[best_model_name]["pred"], zero_division=0))

In [ ]:
log_reg = fitted_models["Logistic Regression"]
log_reg_model = log_reg.named_steps["model"]
coef_df = pd.DataFrame({"feature": modeling_features, "coefficient_scaled": log_reg_model.coef_[0]})
coef_df["odds_ratio_per_1_sd"] = np.exp(coef_df["coefficient_scaled"])
coef_df["abs_coefficient"] = coef_df["coefficient_scaled"].abs()
coef_df["direction"] = np.where(coef_df["coefficient_scaled"] < 0, "negative", "positive")
coef_df = coef_df.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
display(coef_df[["feature", "coefficient_scaled", "odds_ratio_per_1_sd"]])

fig, ax = plt.subplots(figsize=(10, 7))
coef_plot = coef_df.sort_values("coefficient_scaled")
sns.barplot(
    data=coef_plot, x="coefficient_scaled", y="feature", hue="direction",
    palette={"negative": "#2a9d8f", "positive": "#e76f51"},
    dodge=False, legend=False, ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Logistic Regression coefficients")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

catboost_cat_features = [col for col in binary_features if col in modeling_features]

advanced_models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=300, depth=5, learning_rate=0.05, loss_function="Logloss", eval_metric="F1", cat_features=catboost_cat_features, one_hot_max_size=2, auto_class_weights="Balanced", random_seed=42, verbose=False),
}
list(advanced_models.keys())

In [ ]:
advanced_results = []
advanced_fitted_models = {}
advanced_predictions = {}

for name, estimator in advanced_models.items():
    fitted = copy.deepcopy(estimator)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    y_score = fitted.predict_proba(X_test)[:, 1] if hasattr(fitted, "predict_proba") else y_pred
    advanced_fitted_models[name] = fitted
    advanced_predictions[name] = {"pred": y_pred, "score": y_score}
    advanced_results.append({
        "model": name,
        "train_rows": len(X_train),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
    })

advanced_results_df = pd.DataFrame(advanced_results).sort_values("f1", ascending=False).reset_index(drop=True)
display(advanced_results_df)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_train_inner, X_val, y_train_inner, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
optuna_n_trials = 20
neg_pos_ratio = (y_train_inner == 0).sum() / max((y_train_inner == 1).sum(), 1)


def make_advanced_estimator(model_name, trial):
    if model_name == "Random Forest":
        return RandomForestClassifier(n_estimators=trial.suggest_int("n_estimators", 120, 420, step=60), max_depth=trial.suggest_int("max_depth", 5, 18), min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 200, step=20), max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]), class_weight="balanced_subsample", n_jobs=-1, random_state=42)
    if model_name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(max_iter=trial.suggest_int("max_iter", 100, 450, step=50), learning_rate=trial.suggest_float("learning_rate", 0.02, 0.16, log=True), max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 63, step=8), min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 180, step=20), l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 2.0, log=True), random_state=42)
    if model_name == "XGBoost":
        return XGBClassifier(n_estimators=trial.suggest_int("n_estimators", 120, 420, step=60), max_depth=trial.suggest_int("max_depth", 2, 8), learning_rate=trial.suggest_float("learning_rate", 0.02, 0.18, log=True), subsample=trial.suggest_float("subsample", 0.65, 1.0), colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0), min_child_weight=trial.suggest_float("min_child_weight", 1.0, 12.0), reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), scale_pos_weight=neg_pos_ratio, eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1)
    if model_name == "LightGBM":
        return LGBMClassifier(n_estimators=trial.suggest_int("n_estimators", 120, 420, step=60), num_leaves=trial.suggest_int("num_leaves", 15, 63), learning_rate=trial.suggest_float("learning_rate", 0.02, 0.18, log=True), min_child_samples=trial.suggest_int("min_child_samples", 20, 180, step=20), subsample=trial.suggest_float("subsample", 0.65, 1.0), colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0), reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)
    if model_name == "CatBoost":
        return CatBoostClassifier(iterations=trial.suggest_int("iterations", 120, 420, step=60), depth=trial.suggest_int("depth", 3, 8), learning_rate=trial.suggest_float("learning_rate", 0.02, 0.18, log=True), l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 12.0), random_strength=trial.suggest_float("random_strength", 0.1, 10.0, log=True), bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0), border_count=trial.suggest_int("border_count", 32, 128, step=32), loss_function="Logloss", eval_metric="F1", cat_features=catboost_cat_features, one_hot_max_size=2, auto_class_weights="Balanced", random_seed=42, verbose=False)
    raise ValueError(f"Unknown model: {model_name}")


def objective_factory(model_name):
    def objective(trial):
        estimator = make_advanced_estimator(model_name, trial)
        estimator.fit(X_train_inner, y_train_inner)
        y_val_pred = estimator.predict(X_val)
        return f1_score(y_val, y_val_pred, zero_division=0)
    return objective


tuning_studies = {}
tuning_rows = []
for model_name in advanced_models.keys():
    study = optuna.create_study(direction="maximize", study_name=f"{model_name} tuning")
    study.optimize(objective_factory(model_name), n_trials=optuna_n_trials, show_progress_bar=False)
    tuning_studies[model_name] = study
    tuning_rows.append({"model": model_name, "best_validation_f1": study.best_value, "best_params": study.best_params})
tuning_results_df = pd.DataFrame(tuning_rows).sort_values("best_validation_f1", ascending=False)
display(tuning_results_df)

In [ ]:
tuned_advanced_results = []
tuned_advanced_fitted_models = {}
tuned_advanced_predictions = {}

for model_name, study in tuning_studies.items():
    fixed_trial = optuna.trial.FixedTrial(study.best_params)
    fitted = make_advanced_estimator(model_name, fixed_trial)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    y_score = fitted.predict_proba(X_test)[:, 1] if hasattr(fitted, "predict_proba") else y_pred
    tuned_name = f"{model_name} tuned"
    tuned_advanced_fitted_models[tuned_name] = fitted
    tuned_advanced_predictions[tuned_name] = {"pred": y_pred, "score": y_score}
    tuned_advanced_results.append({
        "model": tuned_name,
        "train_rows": len(X_train),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
    })

tuned_advanced_results_df = pd.DataFrame(tuned_advanced_results).sort_values("f1", ascending=False).reset_index(drop=True)
display(tuned_advanced_results_df)

In [ ]:
simple_results_for_compare = results_df.copy()
simple_results_for_compare["model_group"] = "Simple"
advanced_results_for_compare = advanced_results_df.copy()
advanced_results_for_compare["model_group"] = "Advanced baseline"
tuned_results_for_compare = tuned_advanced_results_df.copy()
tuned_results_for_compare["model_group"] = "Advanced tuned"

all_results_df = pd.concat([simple_results_for_compare, advanced_results_for_compare, tuned_results_for_compare], ignore_index=True).sort_values("f1", ascending=False).reset_index(drop=True)
display(all_results_df)

all_results_long = all_results_df.melt(id_vars=["model", "model_group"], value_vars=metric_cols, var_name="metric", value_name="score")
fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(data=all_results_long, x="metric", y="score", hue="model", palette="tab20", ax=ax)
ax.set_title("Simple vs advanced model comparison")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=30)
ax.legend(title="model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
print(f"Best overall model by F1: {all_results_df.iloc[0]['model']}")

In [ ]:
importance_frames = []
importance_models = {}
importance_models.update(advanced_fitted_models)
importance_models.update(tuned_advanced_fitted_models)

for name, fitted in importance_models.items():
    if hasattr(fitted, "feature_importances_"):
        importance_frames.append(pd.DataFrame({"model": name, "feature": modeling_features, "importance": fitted.feature_importances_}))

if importance_frames:
    feature_importance_df = pd.concat(importance_frames, ignore_index=True)
    display(feature_importance_df.sort_values(["model", "importance"], ascending=[True, False]).reset_index(drop=True))
    top_features = feature_importance_df.sort_values("importance", ascending=False).groupby("model").head(12)
    fig, axes = plt.subplots(len(top_features["model"].unique()), 1, figsize=(10, 4 * len(top_features["model"].unique())), squeeze=False)
    for ax, (model_name, tmp) in zip(axes.reshape(-1), top_features.groupby("model")):
        tmp = tmp.sort_values("importance")
        sns.barplot(data=tmp, x="importance", y="feature", hue="feature", palette="viridis", legend=False, ax=ax)
        ax.set_title(f"{model_name}: feature importance")
    plt.tight_layout()
    plt.show()
else:
    print("No tree-based feature_importances_ available for fitted advanced models.")

In [ ]:
import importlib.util

from sklearn.decomposition import PCA, TruncatedSVD, NMF
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler

has_umap = importlib.util.find_spec("umap") is not None
if has_umap:
    import umap.umap_ as umap
else:
    print("UMAP is not installed, skipping UMAP. Install with: pip install umap-learn")

In [ ]:
embedding_sample_size = 6000
if len(X_model) > embedding_sample_size:
    X_dim_sample, _, y_dim_sample, _ = train_test_split(
        X_model, y_model, train_size=embedding_sample_size, random_state=42, stratify=y_model
    )
else:
    X_dim_sample = X_model.copy()
    y_dim_sample = y_model.copy()

X_dim_scaled = StandardScaler().fit_transform(X_dim_sample)
dim_sample_summary = pd.DataFrame({
    "rows": [len(X_dim_sample)],
    "positive_share": [y_dim_sample.mean()],
    "duplicate_rows_kept": [pd.concat([X_dim_sample, y_dim_sample.rename(target)], axis=1).duplicated().sum()],
})
display(dim_sample_summary)

In [ ]:
n_components = min(10, X_dim_scaled.shape[1])

pca = PCA(n_components=n_components, random_state=42)
pca_components = pca.fit_transform(X_dim_scaled)

svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_components = svd.fit_transform(X_dim_scaled)

X_dim_nonnegative = MinMaxScaler().fit_transform(X_dim_sample)
nmf = NMF(n_components=n_components, init="nndsvda", max_iter=1000, random_state=42)
nmf_components = nmf.fit_transform(X_dim_nonnegative)

explained_df = pd.DataFrame({
    "component": np.arange(1, n_components + 1),
    "pca_explained_variance_ratio": pca.explained_variance_ratio_,
    "pca_cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    "svd_explained_variance_ratio": svd.explained_variance_ratio_,
    "svd_cumulative_explained_variance": np.cumsum(svd.explained_variance_ratio_),
})
display(explained_df)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=explained_df, x="component", y="pca_cumulative_explained_variance", marker="o", label="PCA", ax=ax)
sns.lineplot(data=explained_df, x="component", y="svd_cumulative_explained_variance", marker="o", label="SVD", ax=ax)
ax.set_title("Cumulative explained variance")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
projection_frames = []
for method_name, values in {"PCA": pca_components, "SVD": svd_components, "NMF": nmf_components}.items():
    projection_frames.append(pd.DataFrame({
        "method": method_name,
        "component_1": values[:, 0],
        "component_2": values[:, 1],
        target: y_dim_sample.to_numpy(),
    }))
projection_df = pd.concat(projection_frames, ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, method_name in zip(axes, ["PCA", "SVD", "NMF"]):
    tmp = projection_df[projection_df["method"] == method_name]
    sns.scatterplot(data=tmp, x="component_1", y="component_2", hue=target, palette=target_palette, alpha=0.45, s=18, linewidth=0, ax=ax)
    ax.set_title(method_name)
plt.tight_layout()
plt.show()

In [ ]:
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=modeling_features,
    columns=[f"PC{i}" for i in range(1, n_components + 1)],
)
display(pca_loadings[["PC1", "PC2"]].sort_values("PC1", key=lambda s: s.abs(), ascending=False))

fig, ax = plt.subplots(figsize=(9, 6))
loadings_plot = pca_loadings[["PC1", "PC2"]].reset_index().rename(columns={"index": "feature"})
sns.scatterplot(data=loadings_plot, x="PC1", y="PC2", color="#457b9d", s=70, ax=ax)
for _, row in loadings_plot.iterrows():
    ax.text(row["PC1"], row["PC2"], row["feature"], fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("PCA feature loadings")
plt.tight_layout()
plt.show()

In [ ]:
embedding_base = pca_components[:, :min(10, pca_components.shape[1])]

tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", max_iter=1000, random_state=42)
tsne_embedding = tsne.fit_transform(embedding_base)

embedding_frames = [pd.DataFrame({
    "method": "t-SNE",
    "x": tsne_embedding[:, 0],
    "y": tsne_embedding[:, 1],
    target: y_dim_sample.to_numpy(),
})]

if has_umap:
    umap_embedding = umap.UMAP(
        n_components=2, n_neighbors=30, min_dist=0.15, metric="euclidean", random_state=42
    ).fit_transform(embedding_base)
    embedding_frames.append(pd.DataFrame({
        "method": "UMAP",
        "x": umap_embedding[:, 0],
        "y": umap_embedding[:, 1],
        target: y_dim_sample.to_numpy(),
    }))

embedding_df = pd.concat(embedding_frames, ignore_index=True)
methods = embedding_df["method"].unique()
fig, axes = plt.subplots(1, len(methods), figsize=(7 * len(methods), 5), squeeze=False)
for ax, method_name in zip(axes.reshape(-1), methods):
    tmp = embedding_df[embedding_df["method"] == method_name]
    sns.scatterplot(data=tmp, x="x", y="y", hue=target, palette=target_palette, alpha=0.5, s=18, linewidth=0, ax=ax)
    ax.set_title(method_name)
plt.tight_layout()
plt.show()